# NumPy

**Subject:** Introduction to Problem Solving and Reasoning (IPSR)  
**Unit 1 (Python Basics), Topic 16**  
**How to use this notebook:** read each section, run every code cell (Shift + Enter), then change the examples and run them again. Try each exercise before you open its solution.

## 1. Learning goals

After this notebook you can:

* create arrays with `array`, `arange`, `linspace`, `zeros`, `ones` and `reshape`
* read `shape`, `ndim`, `dtype` and `size`
* do fast arithmetic on whole arrays instead of loops
* index and slice arrays, and explain why slices are views
* select values with boolean masks
* use broadcasting and read its shape errors
* sum and average along an axis
* avoid the traps: integer overflow, float precision, views, array equality, empty arrays, `NaN`

## 2. Why NumPy?

A Python list can hold anything, so it is flexible but slow for maths. A **NumPy array** (`ndarray`) holds numbers of **one type** in a tight block of memory.
That makes it much faster, and lets us write maths on whole arrays in one line. Pandas, Matplotlib and most machine learning tools are built on it.

In [1]:
import numpy as np
print(np.__version__)

2.5.3


## 3. Creating arrays

In [2]:
a = np.array([3, 1, 4, 1, 5])
print(a)
print(type(a))

m = np.array([[1, 2, 3],
              [4, 5, 6]])      # a list of lists makes a 2-D array
print(m)

[3 1 4 1 5]
<class 'numpy.ndarray'>
[[1 2 3]
 [4 5 6]]


Ready-made arrays:

* `np.arange(start, stop, step)` like `range`, stop excluded
* `np.linspace(start, stop, n)` exactly `n` evenly spaced numbers, stop **included**
* `np.zeros(shape)`, `np.ones(shape)`, `np.full(shape, value)`

In [3]:
print(np.arange(0, 10, 2))
print(np.arange(5))
print(np.linspace(0, 1, 5))
print(np.zeros(3))
print(np.ones((2, 3)))
print(np.full((2, 2), 7))

[0 2 4 6 8]
[0 1 2 3 4]
[0.   0.25 0.5  0.75 1.  ]
[0. 0. 0.]
[[1. 1. 1.]
 [1. 1. 1.]]
[[7 7]
 [7 7]]


`reshape` changes the shape without changing the data. The total number of items must stay the same. `-1` means "work it out".

In [4]:
r = np.arange(12).reshape(3, 4)
print(r)
print(np.arange(12).reshape(2, -1))     # 2 rows, columns worked out (6)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]


## 4. Shape, dimensions and type

| Attribute | Meaning |
|---|---|
| `shape` | size along each axis, as a tuple |
| `ndim` | number of axes |
| `size` | total number of items |
| `dtype` | the type of every item |

In [5]:
print("shape:", r.shape)
print("ndim :", r.ndim)
print("size :", r.size)
print("dtype:", r.dtype)
print(np.array([1, 2, 3]).dtype)
print(np.array([1.0, 2, 3]).dtype)     # one float makes everything float
print(np.array([1, 2, 3], dtype=float).dtype)
print(np.array([1.7, 2.9]).astype(int))   # astype cuts the decimals, it does not round

shape: (3, 4)
ndim : 2
size : 12
dtype: int64
int64
float64
float64
[1 2]


## 5. Vectorised arithmetic

Operations on arrays work on **every item at once**. No loop is needed. This is called **vectorisation**.
Compare with a list, where `+` joins and `*` repeats.

In [6]:
prices = np.array([100, 250, 80, 40])
print(prices * 2)
print(prices + 5)
print(prices * np.array([1, 2, 3, 4]))     # item by item
print(prices ** 2)
print(np.sqrt(prices))

[200 500 160  80]
[105 255  85  45]
[100 500 240 160]
[10000 62500  6400  1600]
[10.         15.8113883   8.94427191  6.32455532]


In [7]:
as_list = [100, 250, 80, 40]
print(as_list * 2)          # a list is repeated, not multiplied
print(as_list + [5])        # a list is joined

[100, 250, 80, 40, 100, 250, 80, 40]
[100, 250, 80, 40, 5]


Speed is the other reason. Let us time both ways to add up squares of a million numbers.

In [8]:
import time
n = 1_000_000
data = list(range(n))
arr = np.arange(n)

t0 = time.perf_counter()
s1 = sum(x * x for x in data)
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
s2 = int((arr * arr).sum())
t_np = time.perf_counter() - t0

print("same answer:", s1 == s2)
print("NumPy is faster:", t_np < t_list)     # exact times depend on your computer

same answer: True
NumPy is faster: True


Useful reductions: `sum`, `mean`, `min`, `max`, `std`, `cumsum`, `argmax`.

In [9]:
v = np.array([4, 8, 15, 16, 23, 42])
print(v.sum(), v.mean(), v.min(), v.max())
print(v.cumsum())
print(v.argmax())        # position of the largest item

108 18.0 4 42
[  4  12  27  43  66 108]
5


## 6. Indexing and slicing

Same idea as lists: positions start at 0, a slice `a[start:stop:step]` excludes `stop`.
For 2-D arrays write `a[row, column]`.

In [10]:
a = np.arange(10, 100, 10)
print(a)
print(a[0], a[-1])
print(a[2:5])
print(a[::-1])
print(a[[0, 3, 5]])       # a list of positions picks those items

[10 20 30 40 50 60 70 80 90]
10 90
[30 40 50]
[90 80 70 60 50 40 30 20 10]
[10 40 60]


In [11]:
m = np.arange(1, 13).reshape(3, 4)
print(m)
print(m[1, 2])        # row 1, column 2
print(m[0])           # first row
print(m[:, 1])        # every row, column 1
print(m[1:, :2])      # rows 1 and 2, columns 0 and 1

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
7
[1 2 3 4]
[ 2  6 10]
[[ 5  6]
 [ 9 10]]


### Slices are views, not copies
A slice **shares memory** with the original array. Changing the slice changes the original. Use `.copy()` if you want an independent array.

In [12]:
base = np.array([1, 2, 3, 4, 5])
piece = base[1:4]
piece[0] = 99
print(base)               # base changed too
print(piece.base is base) # True: piece is a view of base

safe = base[1:4].copy()
safe[0] = -1
print(base)               # unchanged this time

[ 1 99  3  4  5]
True
[ 1 99  3  4  5]


## 7. Boolean masks

A comparison gives an array of `True` and `False`. Use it to pick items or to change them. Join conditions with `&`, `|`, `~` and put each in brackets.

In [13]:
marks = np.array([78, 45, 91, 33, 60, 85])
mask = marks >= 60
print(mask)
print(marks[mask])
print(marks[(marks >= 40) & (marks < 80)])
print(mask.sum(), "students scored 60 or more")    # True counts as 1

[ True False  True False  True  True]
[78 91 60 85]
[78 45 60]
4 students scored 60 or more


In [14]:
curved = marks.copy()
curved[curved < 40] = 40        # raise low marks to 40
print(curved)
print(np.where(marks >= 40, "pass", "fail"))

[78 45 91 40 60 85]
['pass' 'pass' 'pass' 'fail' 'pass' 'pass']


Unlike slices, a mask (or a list of positions) gives a **copy**.

In [15]:
picked = marks[marks > 80]
picked[0] = 0
print(marks)       # unchanged

[78 45 91 33 60 85]


## 8. Broadcasting

**Broadcasting** lets NumPy combine arrays of different shapes. Rule: compare the shapes from the **right**. Two sizes fit if they are **equal** or one of them is **1**. A size of 1 is stretched to match.

Worked example: add a different bonus to each **column** of marks.

In [16]:
scores = np.array([[50, 60, 70],
                   [55, 65, 75]])           # shape (2, 3)
bonus = np.array([10, 20, 30])              # shape (3,)

print(scores.shape, bonus.shape)
print(scores + bonus)

(2, 3) (3,)
[[ 60  80 100]
 [ 65  85 105]]


Step by step: `(2, 3)` and `(3,)`. From the right: 3 and 3 match. The second array has no second axis, so it is repeated for each row. The bonus row `[10 20 30]` is added to both rows.

To add a different bonus to each **row**, make the bonus a column with shape `(2, 1)`.

In [17]:
row_bonus = np.array([[100],
                      [200]])               # shape (2, 1)
print(scores + row_bonus)
print((np.array([100, 200])[:, np.newaxis]).shape)   # another way to make a column

[[150 160 170]
 [255 265 275]]
(2, 1)


A classic use: a multiplication table from a column and a row.

In [18]:
col = np.arange(1, 4).reshape(3, 1)     # shape (3, 1)
row = np.arange(1, 5)                   # shape (4,)
print(col * row)                        # shape (3, 4)

[[ 1  2  3  4]
 [ 2  4  6  8]
 [ 3  6  9 12]]


If the shapes do not fit, NumPy raises a `ValueError` that names both shapes.

In [19]:
try:
    print(scores + np.array([1, 2]))      # (2, 3) and (2,): 3 vs 2 do not match
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 


## 9. The `axis` argument

For a 2-D array, **axis 0** runs down the rows (so it gives one result per **column**). **Axis 1** runs across the columns (one result per **row**).

In [20]:
t = np.array([[1, 2, 3],
              [4, 5, 6]])
print(t.sum())            # everything
print(t.sum(axis=0))      # per column: [1+4, 2+5, 3+6]
print(t.sum(axis=1))      # per row: [1+2+3, 4+5+6]
print(t.mean(axis=0))
print(t.max(axis=1))

21
[5 7 9]
[ 6 15]
[2.5 3.5 4.5]
[3 6]


In [21]:
print(t.sum(axis=1).shape, t.sum(axis=1, keepdims=True).shape)   # keepdims keeps the axis, with size 1

(2,) (2, 1)


## 10. Edge cases

### 10.1 Integer overflow
Each dtype has a fixed size. `int8` holds -128 to 127 only. Go beyond it and the number **wraps around** with no error.

In [22]:
x = np.array([100, 120], dtype=np.int8)
print(x + 10)             # 110 fits, 130 wraps to -126
print(np.iinfo(np.int8))
print(np.iinfo(np.int64).max)

u = np.array([250], dtype=np.uint8)
print(u + 10)             # wraps to 4
print(np.array([5], dtype=np.uint8) - 10)    # unsigned numbers wrap too

[ 110 -126]
Machine parameters for int8
---------------------------------------------------------------
min = -128
max = 127
---------------------------------------------------------------

9223372036854775807
[4]
[251]


Python's own `int` never overflows. NumPy's default `int64` is large, but not unlimited. Choose a bigger dtype if your numbers can be large.

In [23]:
big = np.array([2**62], dtype=np.int64)
print(big * 2)            # 2**63 does not fit in int64: wraps to a negative number
print(2**62 * 2)          # Python int is fine

[-9223372036854775808]
9223372036854775808


### 10.2 Floating point precision
Decimals are stored in binary, so many cannot be exact. Never compare floats with `==`. Use `np.isclose`.

In [24]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(np.isclose(0.1 + 0.2, 0.3))
print(np.float32(0.1) == np.float64(0.1))   # float32 holds fewer digits
print(np.allclose(np.array([0.1 + 0.2]), np.array([0.3])))

0.30000000000000004
False
True
False
True


### 10.3 Comparing two arrays gives an array
`==` compares item by item and returns an array. Using that in an `if` is ambiguous, so NumPy raises an error.

In [25]:
p = np.array([1, 2, 3])
q = np.array([1, 2, 3])
print(p == q)
try:
    if p == q:
        print("equal")
except ValueError as e:
    print("ValueError:", e)

print((p == q).all())            # all items equal?
print(np.array_equal(p, q))      # same, and also checks the shape

[ True  True  True]
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
True
True


### 10.4 Empty arrays
Some reductions work on an empty array, some do not.

In [26]:
e = np.array([])
print(e.sum(), e.size)           # sum of nothing is 0.0
try:
    print(e.max())
except ValueError as err:
    print("ValueError:", err)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # NumPy warns about the empty mean
    print(e.mean())                   # nan

0.0 0
ValueError: zero-size array to reduction operation maximum which has no identity
nan


### 10.5 `NaN` (missing numbers)
`NaN` means "not a number". It spreads: any normal reduction that touches a `NaN` returns `NaN`. Use the `nan` versions to skip it.

In [27]:
w = np.array([4.0, np.nan, 8.0])
print(w.mean())                  # nan
print(np.nanmean(w))             # (4 + 8) / 2
print(np.nansum(w), np.nanmax(w))
print(np.nan == np.nan)          # False: test with isnan
print(np.isnan(w))
print(w[~np.isnan(w)])           # keep only the real numbers

nan
6.0
12.0 8.0
False
[False  True False]
[4. 8.]


Whole-number arrays cannot hold `NaN`.

In [28]:
try:
    np.array([1, 2, 3])[0] = np.nan
except ValueError as e:
    print("ValueError:", e)

ValueError: cannot convert float NaN to integer


## 11. Quick summary

| Task | Code |
|---|---|
| Create | `np.array([..])`, `np.arange(a, b, s)`, `np.linspace(a, b, n)`, `np.zeros(s)`, `np.ones(s)` |
| Reshape | `a.reshape(r, c)`, `-1` means "work it out" |
| Describe | `a.shape`, `a.ndim`, `a.size`, `a.dtype` |
| Maths | `a + b`, `a * 2`, `np.sqrt(a)` act on every item |
| Index | `a[i]`, `a[r, c]`, `a[:, c]`, `a[i:j]` (slice is a **view**) |
| Mask | `a[a > 5]`, `&`, `|`, `~` (gives a copy) |
| Broadcast | shapes from the right: equal or 1 |
| Axis | `axis=0` per column, `axis=1` per row |
| Traps | fixed-width overflow, `==` on floats, array in `if`, empty `max`, `NaN` (use `nanmean`) |

## 12. Exercises

### Exercise 1

Create an array of the even numbers from 2 to 20 (include 20) using `arange`. Print it, its `shape` and its `dtype`.

In [29]:
# Write your answer here

<details>
<summary><b>Show solution 1</b></summary>

`arange` excludes the stop value, so use 21.

```python
a = np.arange(2, 21, 2)
print(a, a.shape, a.dtype)
```

</details>

### Exercise 2

A shop has prices in rupees `[120, 85, 40, 260, 15]`. Use arithmetic on the whole array to add 18% tax and print the new prices rounded to 2 decimals.

*Hint:* multiply by 1.18, then `np.round(x, 2)`.

In [30]:
# Write your answer here

<details>
<summary><b>Show solution 2</b></summary>

Expected: `[141.6 100.3  47.2 306.8  17.7]`.

```python
prices = np.array([120, 85, 40, 260, 15])
print(np.round(prices * 1.18, 2))
```

</details>

### Exercise 3

Make the array `np.arange(1, 13).reshape(3, 4)`. Print its second column, its last row, and the 2 by 2 block in the top right corner.

In [31]:
# Write your answer here

<details>
<summary><b>Show solution 3</b></summary>

Second column: `[2 6 10]`. Last row: `[9 10 11 12]`. Top right block: `[[3 4] [7 8]]`.

```python
m = np.arange(1, 13).reshape(3, 4)
print(m[:, 1])
print(m[-1])
print(m[:2, 2:])
```

</details>

### Exercise 4

For `marks = np.array([78, 45, 91, 33, 60, 85, 52])` print how many students scored below 50 and the average of the students who scored 50 or more.

In [32]:
# Write your answer here

<details>
<summary><b>Show solution 4</b></summary>

Two students are below 50. The average of 78, 91, 60, 85 and 52 is 73.2.

```python
marks = np.array([78, 45, 91, 33, 60, 85, 52])
print((marks < 50).sum())
print(marks[marks >= 50].mean())
```

</details>

### Exercise 5

A `(3, 3)` array holds the daily sales of 3 shops over 3 days: `[[10, 20, 30], [5, 15, 25], [8, 9, 10]]`. Print the total per shop, the average per day, and then subtract each day's average from the sales using broadcasting.

*Hint:* per shop means `axis=1`. Per day means `axis=0`.

In [33]:
# Write your answer here

<details>
<summary><b>Show solution 5</b></summary>

Shop totals: `[60 45 27]`. Day averages: about `[7.67 14.67 21.67]`. The `(3, 3)` array minus a `(3,)` array is broadcast row by row.

```python
sales = np.array([[10, 20, 30], [5, 15, 25], [8, 9, 10]])
print(sales.sum(axis=1))
day_avg = sales.mean(axis=0)
print(day_avg)
print(sales - day_avg)
```

</details>

### Exercise 6

**Edge case.** Make `a = np.arange(6)` and `b = a[:3]`. Set `b[0] = 100` and print `a`. Then repeat with `b = a[:3].copy()` on a fresh `a`. Explain the difference in a comment.

In [34]:
# Write your answer here

<details>
<summary><b>Show solution 6</b></summary>

```python
a = np.arange(6)
b = a[:3]
b[0] = 100
print(a)          # [100 1 2 3 4 5]: slice is a view

a = np.arange(6)
b = a[:3].copy()
b[0] = 100
print(a)          # [0 1 2 3 4 5]: copy is independent
```

</details>

### Exercise 7

**Edge case.** Given `d = np.array([2.0, np.nan, 6.0, np.nan, 4.0])`, print the plain `mean`, the `nanmean`, and how many values are missing.

In [35]:
# Write your answer here

<details>
<summary><b>Show solution 7</b></summary>

The plain mean is `nan`. `nanmean` is (2 + 6 + 4) / 3 = 4.0. Two values are missing.

```python
d = np.array([2.0, np.nan, 6.0, np.nan, 4.0])
print(d.mean())
print(np.nanmean(d))
print(np.isnan(d).sum())
```

</details>

## 13. Where to read more

* Python Data Science Handbook, *Introduction to NumPy* (overview page): https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html
* NumPy documentation, *NumPy: the absolute basics for beginners*: https://numpy.org/doc/stable/user/absolute_beginners.html